[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/EelcoHoogendoorn/numga/blob/main/examples/mechanics/whitney_elasticity/whitney_elasticity.ipynb)

# Seismic Waves in a Small Planet

This example computes the seismic waves that a sudden blow sends through a small planet, a liquid core inside a solid shell with a free surface. A blow just below the surface sends out three kinds of wave. Compressional P waves are the fastest and travel through solid and liquid alike, bending as they cross into the slower core. Shear S waves are slower, and a liquid cannot carry them, so the core casts a shadow that S waves reach only around it, through the shell. The surface wave is slower still: it rolls the free surface and stays close to it, so that in a cross-section it runs around the planet without spreading into the interior, and it soon carries most of the motion. The model is a plane-strain cross-section, with small motions and no gravity.

Whitney corner hats carry the displacement field, with a constant strain on each triangle. A corner's gradient, `sides / (2 * patches)`, the opposite side over twice the triangle's patch, is a plane that reads a displacement `d` of that corner as `gradient & d`. The material is a work form on pairs of displacement gradients; binding each triangle's corner gradients into it pulls it back to a stiffness, a map from corner displacements to forces, and the liquid is the same form without a shear modulus, made slightly viscous by the form's shear part applied to the velocity. The two kinds of body wave are told apart by two readings of the velocity: its rate of dilation is the corner gradients paired with the corner velocities and needs no metric, while its rotation wedges the gradients against the metric duals of the velocities, since a turn is measured by angles.

In the potential notation of seismology, the displacement splits as $\mathbf u=\nabla\phi+\nabla\times\boldsymbol\psi$, its two parts travelling at the P and S speeds $\alpha=\sqrt{(\lambda+2\mu)/\rho}$ and $\beta=\sqrt{\mu/\rho}$; the two panels show $\nabla\cdot\dot{\mathbf u}$ and the component of $\nabla\times\dot{\mathbf u}$ across the section. A liquid has $\beta=0$, P waves that cross the core are the phases PKP, and the surface wave is a Rayleigh wave, with speed $c_R\approx0.92\,\beta$ where $\lambda=\mu$.

In [ ]:
# In Colab, numga and its examples, from GitHub.
import sys

if "google.colab" in sys.modules:
    %pip install -q "numga[examples] @ git+https://github.com/EelcoHoogendoorn/numga"


In [ ]:
%matplotlib inline
from collections.abc import Iterable, Iterator
from itertools import chain, repeat
import numpy as np

from numga import stack, where
from numga.algebras import VGA2D as ga
from numga.sparse import SparseExtensor, spdiag
from examples import instantiate
from examples.mechanics.whitney_elasticity import render
from examples.util.animation import save_figure, show_animation

mesh_module = instantiate("examples.util.mesh", ga)
Mesh, as_scalar, mv = mesh_module.Mesh, mesh_module.as_scalar, mesh_module.mv
Scalar = ga.gatype.scalar()                                      # Scalar+
Point = ga.gatype.antivector()                                   # Antivector-
Plane = ga.gatype.vector()                                       # Vector+
Force = ga.gatype.vector()                                       # Vector+
Patch = ga.gatype.antibivector()                                 # Antibivector-
Acceleration = ga.gatype((Point, Point))
Stiffness = ga.gatype((Force, Point))
Elastic = ga.gatype((Scalar, Plane, Plane, Point, Point))

## A liquid core in a solid shell

The cross-section is one mesh of triangles in two regions, a liquid core and a solid shell whose outer rim is the planet's free surface. Liquid and solid share the vertices where they meet: the liquid is slightly viscous, so at its boundary it moves with the shell. The core is denser than the shell and carries P waves more slowly, three times the shell's P-wave impedance, so that a wave arriving at the core is partly reflected and partly passed into it.

In [ ]:
core_radius = 0.55
surface_radius = 1.0
n_radial_divisions = 64
# The liquid core first, then the solid shell; the liquid has no shear stiffness.
layer_densities = np.array([5.0, 1.0])
layer_lambdas = np.array([5.0, 1.0])
layer_mus = np.array([0.0, 1.0])
# The liquid's shear viscosity; the shell's damping is its viscoelasticity alone.
layer_viscosities = np.array([0.001, 0.0])

In [ ]:
mesh, vertices = Mesh.concentric_disk(np.array([core_radius, surface_radius]), n_radial_divisions)
centres = (mesh.incidence[2, 0].mean(axis=0) * vertices).batch()  # [faces] Point-
# The liquid core is region zero, the solid shell region one.
regions = (centres.norm() > core_radius).astype(int)

density = as_scalar(layer_densities[regions]).field()           # Scalar+[faces]
lame_lambda = as_scalar(layer_lambdas[regions])                 # [faces] Scalar+
lame_mu = as_scalar(layer_mus[regions])                         # [faces] Scalar+

save_figure(render.draw_materials(mesh, vertices, regions), "whitney_elasticity_planet_materials")

## Strain energy and stiffness

A triangle's affine hats make its displacement gradient constant. The shell stores energy in dilation and symmetric stretch; the liquid stores only dilation energy. A rigid translation or an infinitesimal rigid turn costs no elastic energy.

In displacement finite-element notation, these are constant-strain triangles, with energy $\frac12\int\varepsilon(u):C:\varepsilon(u)\,dA$ and $\varepsilon(u)=\frac12(\nabla u+\nabla u^T)$.

In [ ]:
# --- math -----------------------------------------------------------------------------
sides = mesh.boundary[2, 1] * (mesh.d[0] * vertices)            # [corners] Point-[faces]
patches = (sides[0] & sides[1]) / 2                             # Patch-[faces]
areas = patches.norm()                                          # Scalar+[faces]
# The opposite edge over twice the oriented patch reads a corner hat's rise.
corner_gradients = sides / (2 * patches)                        # [corners] Plane+[faces]

dilation_traction = Plane * (Plane & Point)
# The exterior-product term removes rotation, leaving the two symmetric shear readings.
direct_shear = (Plane | Plane) * Point.dual_inverse()
shear_traction = 2 * direct_shear - (Plane | (Plane ^ Point.dual_inverse()))
traction = lame_lambda * dilation_traction + lame_mu * shear_traction
material = traction & Point                                    # [faces] Scalar+ <- (Plane+, Plane+, Point-, Point-)



def assemble(material: Elastic) -> Stiffness:
    """Bind the local test and trial gradients: a work form on gradients becomes a form on the corner motions."""
    gradients = corner_gradients.batch()                        # [corners, faces] Plane+
    work = material[None, None](gradients[:, None], gradients[None, :], Point, Point)
    local = (Force & Point).solve(work) * areas.batch()
    return SparseExtensor.from_indices(local, mesh.faces.T[:, None, :], mesh.faces.T[None, :, :],
                                       (mesh.n_vertices, mesh.n_vertices))


stiffness = assemble(material)                                  # (Force+ <- Point-)[vertices <- vertices]

# Each corner hat owns one third of its triangle's mass.
thirds = mesh.incidence[2, 0].mean(axis=0)                      # Scalar+[faces <- vertices]
masses = ~thirds * (density * areas)                            # Scalar+[vertices]
inverse_inertia = spdiag((1 / masses) * Force.dual())            # (Point- <- Force+)[vertices <- vertices]
acceleration_map = -inverse_inertia(stiffness)                  # (Point- <- Point-)[vertices <- vertices]

divergence = (spdiag(corner_gradients & Point) * mesh.incidence[2, 0]).sum(axis=0)               # (Scalar+ <- Point-)[faces <- vertices]
rotation = (spdiag(corner_gradients ^ Point.dual_inverse()) * mesh.incidence[2, 0]).sum(axis=0)  # (Bivector+ <- Point-)[faces <- vertices]

## A slightly viscous liquid

The liquid stores no shear, but it resists the rate of shear: its viscosity is the shear part of the same work form, applied to the velocity instead of the displacement. Assembled like the stiffness, it drags each triangle's rate of shear. A shear wave reaching the core still dies within a short distance, and the slow swirling motions of the liquid, which no stiffness holds back, are damped.

In the notation of continuum mechanics the liquid's stress is $\sigma=\lambda\,(\nabla\cdot\mathbf u)\,I+2\eta\,\dot\varepsilon$, and a shear disturbance of frequency $\omega$ decays over the depth $\delta=\sqrt{2\eta/(\rho\omega)}$.

In [ ]:
viscosity = (as_scalar(layer_viscosities[regions]) * shear_traction) & Point   # [faces] Scalar+ <- (Plane+, Plane+, Point-, Point-)
damping = assemble(viscosity)                                   # (Force+ <- Point-)[vertices <- vertices]
drag_map = -inverse_inertia(damping)                            # (Point- <- Point-)[vertices <- vertices]

## A blow below the free surface

A small moon strikes the surface at an angle and rebounds with part of its normal speed reversed and part of its tangential speed kept. The momentum it gives up is the blow, a short push along that momentum just below the free surface, a smooth bump of velocity several triangles wide, so that it puts no energy into waves as short as the triangles, which the mesh carries too slowly and scatters. A source this shallow puts most of its energy into the surface wave, and an oblique one sends different waves to either side. The material is mildly viscoelastic, its stiffness applied to `displacement + damping_time * velocity`, which damps each wave at a rate proportional to the square of its frequency, so that what little motion reaches the scale of the triangles dies within a few periods while the seismic waves keep most of their amplitude.

In [ ]:
# The place of the blow, on the diagonal towards the upper left.
impact_angle = np.radians(45)
impact_normal = (mv.xy * (-impact_angle / 2)).exp() >> mv.y.dual()   # [] Point-
# The moon: its radius, its speed, the angle of its approach from the surface's normal, the shares of its normal
# speed it rebounds with and of its tangential speed it keeps, and the frames it takes to come in.
moon_radius = 0.07
moon_speed = 2.0
approach_angle = np.radians(40)
restitution = 0.5
sliding = 0.6
n_approach_frames = 12
source_depth = 0.05
source_width = 0.07
source_support = 3.0
source_speed = 0.05
damping_time = 3e-4
dt = 0.00448
n_frames, n_substeps = 100, 8
frame_duration_ms = 40
displacement_scale = 80.0
dilation_scale, vorticity_scale = 0.1, 0.3

In [ ]:
incoming = moon_speed * ((mv.xy * (approach_angle / 2)).exp() >> -impact_normal)   # [] Point-
normal_part = impact_normal * incoming.dot(impact_normal)                    # [] Point-
outgoing = sliding * (incoming - normal_part) - restitution * normal_part     # [] Point-
# The planet's blow is the momentum the moon gives up.
blow = (incoming - outgoing).normalized()                                    # [] Point-
contact = impact_normal * (surface_radius + moon_radius)                      # [] Point-
times = n_substeps * dt * np.arange(-n_approach_frames, n_frames + 1)
moon = contact + where(times < 0, incoming * times, outgoing * times)        # [frames] Point-

source_centre = impact_normal * (surface_radius - source_depth)  # [] Point-
offsets = vertices.batch() - source_centre                      # [vertices] Point-
source_radius_squared = offsets.norm_squared() / source_width**2
taper = (1 - source_radius_squared / source_support**2).squared()
gaussian = (-source_radius_squared / 2).exp()
envelope = where(source_radius_squared < source_support**2, gaussian * taper, 0 * gaussian)
velocity = (source_speed * envelope * blow).field()              # Point-[vertices]
displacement = 0 * velocity                                    # Point-[vertices]

## P, S and surface waves

The left panel colours the rate of dilation, red where the material compresses and blue where it expands; the right panel colours the vorticity, purple where the material turns counter-clockwise and green where it turns clockwise. Each keeps one scale throughout. The P front runs ahead on the left and into the core, where it slows, bends and gathers to a focus; the S front follows on the right and stops at the core, which stays blank, the shadow of the S waves. The surface wave is the strong pair of crests and troughs running along the rim in both panels, and its two halves meet on the far side. Beads ride on the free surface, and the motion is enlarged for display.

This cell takes about six seconds, most of it the time steps and the drawing.

In [ ]:
def evolve(acceleration_map: Acceleration, drag_map: Acceleration, displacement: Point, velocity: Point,
           damping_time: float, dt: float, n_frames: int, n_substeps: int) -> Iterator[tuple[Point, Point]]:
    """Velocity Verlet advances the viscoelastic field, yielding displacement and velocity at frame times."""
    acceleration = acceleration_map(displacement + damping_time * velocity) + drag_map(velocity)
    yield displacement, velocity
    for _ in range(n_frames):
        for _ in range(n_substeps):
            velocity = velocity + (dt / 2) * acceleration
            displacement = displacement + dt * velocity
            # The rate of strain is read from the half-step velocity, which keeps the step explicit.
            acceleration = acceleration_map(displacement + damping_time * velocity) + drag_map(velocity)
            velocity = velocity + (dt / 2) * acceleration
        yield displacement, velocity


def wave_readings(states: Iterable[tuple[Point, Point]]) -> Iterator[tuple[Point, Scalar, Scalar]]:
    """The rate of dilation and the vorticity accompany each moving field to the drawing."""
    for displacement, velocity in states:
        # Each corner gradient reads its corner's velocity: their sum is the rate at which the triangle swells.
        dilation_rate = divergence(velocity)                    # Scalar+[faces]
        vorticity = rotation(velocity) & mv.xy.dual()           # Scalar+[faces]
        yield displacement, dilation_rate, vorticity


states = list(evolve(acceleration_map, drag_map, displacement, velocity, damping_time, dt, n_frames, n_substeps))
# The planet rests while the moon comes in.
frames = chain(repeat((displacement, displacement), n_approach_frames), states)
show_animation(render.animate_waves(mesh, vertices, regions, wave_readings(frames), moon, moon_radius,
                                   displacement_scale, dilation_scale, vorticity_scale),
               "whitney_elasticity_planet", frame_duration_ms, n_frames=n_approach_frames + n_frames + 1)

## Normal modes and travelling waves

The free oscillations of a body are the motions that keep their shape and only swing in amplitude: the eigenvectors of the stiffness against the masses. They are clearest in a uniform ball, where nothing but its shape tells one mode from another, here a ball of cork: it does not spread sideways when squeezed, so its P waves run only the square root of two faster than its S waves, and many of its waves both compress and shear. Most modes come in pairs of one frequency that differ by a turn of the ball, and a pair swung a quarter cycle apart is a wave that travels round it, one wavelength per cycle. Each wave runs through one cycle in turn, lowest first, its dilation on the left and its rotation on the right, on one scale. Surface waves ripple the rim and leave the middle still; volume waves compress and shear through the whole ball.

In the notation of structural dynamics the modes solve $K\phi=\omega^2M\phi$, and a degenerate pair makes the travelling wave $\cos(\omega t)\,\phi_1+\sin(\omega t)\,\phi_2$.

In [ ]:
# Cork's Lame parameters and density, its rigid motions, the modes found, how close two frequencies are when their modes differ by a turn of the ball, the
# travelling waves shown, the frames of each one's cycle, and the largest displacement they are shown at.
ball_lambda, ball_mu, ball_density = 0.0, 1.0, 1.0
n_rigid_motions = 3
n_modes = 40
pair_tolerance = 1e-3
n_travelling_waves = 12
n_cycle_frames = 24
mode_peak = 0.0006

In [ ]:
# A uniform ball of cork, drawn as the shell's region everywhere.
solid = np.ones_like(regions)                                   # [faces]
ball_material = (as_scalar(ball_lambda) * dilation_traction + as_scalar(ball_mu) * shear_traction) & Point   # [] Scalar+ <- (Plane+, Plane+, Point-, Point-)
ball_stiffness = assemble(ball_material)                        # (Force+ <- Point-)[vertices <- vertices]
ball_masses = ~thirds * (as_scalar(ball_density) * areas)       # Scalar+[vertices]
# The dual that takes a force to a point makes the stiffness a map of displacements to themselves, as the masses are.
to_points = spdiag(as_scalar(np.ones(mesh.n_vertices)).field() * Force.dual())   # (Point- <- Force+)[vertices <- vertices]
squared, shapes = to_points(ball_stiffness).eigh(spdiag(ball_masses * Point), n_rigid_motions + n_modes)   # [modes] Scalar+, [modes] Point-[vertices]
squared, shapes = squared[n_rigid_motions:], shapes[n_rigid_motions:]
# Two modes that differ by a turn of the ball share their frequency; together they make a wave that travels round it.
paired = np.flatnonzero((squared[1:] - squared[:-1]) < pair_tolerance * squared[1:])[:n_travelling_waves]


def travelling(first: Point, second: Point, peak: float, n_cycle_frames: int) -> Iterator[tuple[Point, Scalar, Scalar]]:
    """Each pair's wave through one cycle, the first mode a quarter cycle ahead of the second, its largest displacement
    `peak`, with the dilation and the rotation it makes."""
    for pair in range(len(first)):
        sizes = first[pair].norm().batch()                          # [vertices] Scalar+
        scale = peak / sizes[sizes.argmax()]                        # [] Scalar+
        for phase in np.linspace(0, 2 * np.pi, n_cycle_frames, endpoint=False):
            displacement = scale * (np.cos(phase) * first[pair] + np.sin(phase) * second[pair])   # Point-[vertices]
            yield displacement, divergence(displacement), rotation(displacement) & mv.xy.dual()


show_animation(render.animate_modes(mesh, vertices, solid, travelling(shapes[paired], shapes[paired + 1], mode_peak,
                                                                      n_cycle_frames), displacement_scale),
               "whitney_elasticity_modes", frame_duration_ms, n_frames=n_travelling_waves * n_cycle_frames)

In [ ]:
# checks
translations = vertices * 0 + mv.x.dual()
np.testing.assert_allclose(stiffness(translations).kernel, 0.0, atol=1e-11)
np.testing.assert_allclose(stiffness(vertices.commutator(mv.xy)).kernel, 0.0, atol=1e-11)


def energy(displacement: Point, velocity: Point, stiffness: Stiffness, masses: Scalar) -> Scalar:
    """Kinetic and elastic energy of the discrete field."""
    kinetic = (masses * (velocity.dual_inverse() & velocity)).sites.sum() / 2
    elastic = (stiffness(displacement) & displacement).sites.sum() / 2
    return kinetic + elastic


initial_momentum = (masses * velocity).sites.sum()
spin_motion = vertices.commutator(mv.xy)
initial_spin = (masses * (spin_motion.dual_inverse() & velocity)).sites.sum()
energies = []
for last_displacement, last_velocity in states:
    energies.append(energy(last_displacement, last_velocity, stiffness, masses))
energy_fractions = (stack(energies) / energies[0]).kernel
# The damping only takes energy away, so the energy falls from frame to frame, and the waves keep most of it.
assert np.all(np.diff(energy_fractions) < 0)
assert energy_fractions[-1] > 0.5
# Elastic, damping and viscous forces are internal, so they keep the momentum and the angular momentum.
np.testing.assert_allclose(((masses * last_velocity).sites.sum() - initial_momentum).kernel, 0.0, atol=1e-11)
np.testing.assert_allclose(((masses * (spin_motion.dual_inverse() & last_velocity)).sites.sum() - initial_spin).kernel,
                           0.0, atol=1e-11)